# Two-Stage Approach: Low-Param Channel Estimation → Classical Angle Extraction

```
Y  →  [small CNN]  →  Ĥ  →  [2D-DFT+SIC on H]  →  (AoA, AoD)
      (learns channel space, NOT angle-heatmap space)
```

**Paper approach:** Y → [64-block ResNet, 469K params] → 256×256 heatmap → blob → angles
**এই approach:** Y → [small CNN, 16×16 output] → Ĥ → classical extraction → angles

Output space 256× ছোট (16×16 vs 256×256) → network অনেক ছোট হতে পারে, training অনেক দ্রুত হবে।

### দুটো metric level
1. **NMSE(Ĥ, H)** — channel estimation quality, broader DL-channel-estimation literature-এর সাথে তুলনীয়
2. **Pd / RMSE** — downstream angle accuracy, এই paper-এর ResNet/U-Net-এর সাথে তুলনীয় (same metric, same protocol: L=3, Q=P=16)

### Time budget
Training config নিচে ~১-২ ঘণ্টা target করে বানানো (default 300 epoch)। একটা **hard safety cap ৩ ঘণ্টায়** — এর বেশি গেলে training নিজে থেকে থেমে যাবে, checkpoint দিয়েই evaluate হবে।

In [ ]:
# Cell 1 — Setup
import importlib, subprocess, sys
try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'], check=True)

import os, math, time, json
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from scipy.ndimage import maximum_filter
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Input, BatchNormalization, Activation, Add
from tensorflow.keras.models import Model

tf.get_logger().setLevel('ERROR')
np.random.seed(42); tf.random.set_seed(42)

gpus = tf.config.list_physical_devices('GPU')
print(f'TF: {tf.__version__}  |  GPU: {gpus}')
if gpus:
    for g in gpus: tf.config.experimental.set_memory_growth(g, True)
    strategy = tf.distribute.MirroredStrategy()
    print(f'✅ {strategy.num_replicas_in_sync} GPU(s)')
else:
    strategy = tf.distribute.get_strategy()
    print('⚠️ No GPU')

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Cell 2 — Physics (paper-exact) + data generator (NATIVE resolution, no interpolation)
#
# Scope decision: P=Q=nt=nr=16 fixed for this study (paper's evaluation condition).
# Unlike the heatmap-ResNet, Y is NOT upsampled here -- Y and H are the same size
# (16x16), so the CNN is a plain same-resolution residual regressor, no transpose-conv
# upsampling stage needed at all. This is the main reason it can be so much smaller.

class _H(np.ndarray):
    @property
    def H(self): return self.conj().transpose()

def ev(n, angle):
    return ((1/np.sqrt(n)) * np.exp(-1j*np.pi*np.cos(angle)*np.arange(n))).reshape(-1,1)

def make_F(P, nt):
    phi = np.arccos((1/np.pi)*np.angle(np.exp( 1j*(2*np.pi/P)*np.arange(P))))
    F = np.zeros((nt, P), dtype=complex)
    for i, ph in enumerate(phi): F[:, i] = ev(nt, ph).ravel()
    return F

def make_W(Q, nr):
    phi = np.arccos((1/np.pi)*np.angle(np.exp(-1j*(2*np.pi/Q)*np.arange(Q))))
    W = np.zeros((nr, Q), dtype=complex)
    for i, ph in enumerate(phi): W[:, i] = ev(nr, ph).ravel()
    return W

def gen_channel(nr, nt, phi_l, psi_l, alpha_l):
    Hm = np.zeros((nr, nt), dtype=complex)
    for a, phi, psi in zip(alpha_l, phi_l, psi_l):
        Hm += a * (ev(nr, psi) * ev(nt, phi).view(_H).H)
    return np.sqrt(nt * nr) * Hm

def gen_points(L, delta=np.pi/6, max_try=20000):
    pts = []
    for _ in range(max_try):
        if len(pts) == L: break
        x, y = np.random.uniform(0, np.pi), np.random.uniform(0, np.pi)
        if all(math.hypot(x-p[0], y-p[1]) >= delta for p in pts): pts.append((x, y))
    if len(pts) < L: raise RuntimeError('point placement failed')
    return pts

P = Q = nt = nr = 16   # fixed for this study
F_FIXED = make_F(P, nt)
W_FIXED = make_W(Q, nr)

def data_gen_channel(Training=True, condition=None):
    """Yields (Y_16x16x2, H_16x16x2) for training, or (Y, H, feat) for eval."""
    while True:
        if Training:
            L = np.random.randint(1, 10)
            SNR = np.random.randint(-15, 25)
        else:
            L, SNR = condition

        alpha = (np.sqrt(1/L)/np.sqrt(2)) * (np.random.randn(L) + 1j*np.random.randn(L))
        alpha = alpha[np.argsort(-np.abs(alpha))]
        pts = gen_points(L)
        phi_l = np.array([p[0] for p in pts]); psi_l = np.array([p[1] for p in pts])

        Hm = gen_channel(nr, nt, phi_l, psi_l, alpha)
        var = 10**(-SNR/10); s = np.sqrt(var/2)
        Z = s * (np.random.randn(Q, P) + 1j*np.random.randn(Q, P))
        Y = (W_FIXED.view(_H).H @ Hm) @ F_FIXED + Z

        Y_in = np.stack([Y.real, Y.imag], axis=-1).astype(np.float32)    # (16,16,2) -- NATIVE, no zoom
        H_out = np.stack([Hm.real, Hm.imag], axis=-1).astype(np.float32) # (16,16,2)

        if Training:
            yield Y_in, H_out
        else:
            yield Y_in, H_out, np.stack([psi_l, phi_l]).astype(np.float32)

print('✅ Physics + native-resolution data generator ready (P=Q=nt=nr=16, no upsampling)')

In [ ]:
# Cell 3 — Low-parameter channel-estimation CNN (same-resolution residual regressor)
def res_conv_small(x, f):
    skip = x
    x = Conv2D(f, 3, padding='same')(x); x = BatchNormalization()(x); x = Activation('relu')(x)
    x = Conv2D(f, 3, padding='same')(x); x = BatchNormalization()(x)
    x = Add()([x, skip]); x = Activation('relu')(x)
    return x

def build_channel_estimator(n_blocks=6, filters=16):
    x_in = Input(shape=(16, 16, 2))
    x = Conv2D(filters, 3, padding='same')(x_in)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    for _ in range(n_blocks):
        x = res_conv_small(x, filters)
    x = Conv2D(2, 3, padding='same')(x)     # linear output -- regression, no activation
    return Model(x_in, x, name=f'ChannelEstCNN-{n_blocks}b-{filters}f')

N_BLOCKS = 6
FILTERS  = 16

with strategy.scope():
    ce_model = build_channel_estimator(N_BLOCKS, FILTERS)
    ce_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')

PAPER_RESNET_PARAMS = 469393
my_params = ce_model.count_params()
print(f'This model:    {my_params:,} params')
print(f'Paper ResNet:  {PAPER_RESNET_PARAMS:,} params')
print(f'Ratio:         {my_params/PAPER_RESNET_PARAMS:.1%}  ({PAPER_RESNET_PARAMS/my_params:.1f}x smaller)')
ce_model.summary(line_length=80)

In [ ]:
# Cell 4 — Training config + hard time-budget safety cap
BATCH           = 32
STEPS_PER_EPOCH = 312
EPOCHS          = 300              # target ~1-2h; adjust down if needed
MAX_HOURS       = 3.0              # hard cap -- training stops itself if exceeded

class TimeBudgetCallback(tf.keras.callbacks.Callback):
    def __init__(self, max_hours):
        super().__init__()
        self.max_seconds = max_hours * 3600
        self.start = None
    def on_train_begin(self, logs=None):
        self.start = time.time()
    def on_epoch_end(self, epoch, logs=None):
        elapsed = time.time() - self.start
        if elapsed > self.max_seconds:
            print(f'\n⏱️  Time budget ({self.max_seconds/3600:.1f}h) hit at epoch {epoch+1} — stopping, '
                  f'best checkpoint will still be used for evaluation.')
            self.model.stop_training = True

WEIGHTS_PATH = os.path.join(OUT_DIR, 'channel_est_best.weights.h5')

print(f'Params: {my_params:,}  ({PAPER_RESNET_PARAMS/my_params:.1f}x smaller than paper ResNet)')
print(f'BATCH={BATCH}  STEPS={STEPS_PER_EPOCH}  EPOCHS={EPOCHS}  hard cap={MAX_HOURS}h')
print('Output is 16x16 (256 complex values) vs 256x256 (65536 values) for the heatmap approach,')
print('and there is no upsampling stage -- expect well under an hour on a T4, but the cap is there')
print('as a hard backstop regardless.')

In [ ]:
# Cell 5 — Train
def make_train_ds(batch):
    gen = data_gen_channel(Training=True)
    def fn():
        for y, h in gen: yield y, h
    ds = tf.data.Dataset.from_generator(
        fn,
        output_signature=(
            tf.TensorSpec(shape=(16, 16, 2), dtype=tf.float32),
            tf.TensorSpec(shape=(16, 16, 2), dtype=tf.float32),
        )
    )
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(WEIGHTS_PATH, save_best_only=True,
                                       save_weights_only=True, monitor='loss', verbose=0),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.CSVLogger(os.path.join(OUT_DIR, 'channel_est_train_log.csv')),
    TimeBudgetCallback(MAX_HOURS),
]

print(f'Training {ce_model.name} | {EPOCHS} epochs x {STEPS_PER_EPOCH} steps x batch {BATCH}')
t0 = time.time()
history = ce_model.fit(
    make_train_ds(BATCH),
    steps_per_epoch=STEPS_PER_EPOCH,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)
elapsed = time.time() - t0
print(f'\n✅ Training done in {elapsed/60:.1f} min ({elapsed/3600:.2f}h)  |  best loss: {min(history.history["loss"]):.6f}')

ce_model.load_weights(WEIGHTS_PATH)
print('Best weights loaded for evaluation.')

In [ ]:
# Cell 6 — Training curve
plt.figure(figsize=(9, 3.5))
plt.plot(history.history['loss'], color='teal', linewidth=1.5)
plt.xlabel('Epoch'); plt.ylabel('MSE (real/imag)'); plt.title(f'{ce_model.name} Training Loss')
plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'channel_est_training_curve.png'), dpi=150)
plt.show()

In [ ]:
# Cell 7 — Stage 1 metric: NMSE(H_hat, H_true) vs SNR, CNN vs zero-training LS baseline
def nmse(H_true_c, H_est_c):
    num = np.sum(np.abs(H_est_c - H_true_c)**2)
    den = np.sum(np.abs(H_true_c)**2)
    return num / den

# LS baseline: algebraic inversion, zero training -- shows what the CNN adds on top
F_inv    = np.linalg.pinv(F_FIXED)
WH_inv   = np.linalg.pinv(W_FIXED.conj().T)

def ls_estimate(Y_complex):
    return WH_inv @ Y_complex @ F_inv

SNRS_EVAL = list(range(-10, 30, 5))
N_PER_SNR = 500
L_EVAL = 3

nmse_cnn, nmse_ls = {}, {}
for snr in SNRS_EVAL:
    gen = data_gen_channel(Training=False, condition=(L_EVAL, snr))
    errs_cnn, errs_ls = [], []
    for _ in range(N_PER_SNR):
        Y_in, H_out, feat = next(gen)
        Y_complex = Y_in[...,0] + 1j*Y_in[...,1]
        H_true_c  = H_out[...,0] + 1j*H_out[...,1]

        H_pred = ce_model(tf.expand_dims(Y_in, 0), training=False)[0].numpy()
        H_pred_c = H_pred[...,0] + 1j*H_pred[...,1]

        H_ls_c = ls_estimate(Y_complex)

        errs_cnn.append(nmse(H_true_c, H_pred_c))
        errs_ls.append(nmse(H_true_c, H_ls_c))

    nmse_cnn[snr] = np.mean(errs_cnn)
    nmse_ls[snr]  = np.mean(errs_ls)
    print(f'SNR={snr:4d}dB   NMSE(CNN)={10*np.log10(nmse_cnn[snr]):7.2f} dB   '
          f'NMSE(LS)={10*np.log10(nmse_ls[snr]):7.2f} dB')

print('\n✅ NMSE evaluation complete (lower/more-negative dB = better)')

In [ ]:
# Cell 8 — Angle extraction from H_hat (reuses the classical-baseline sinusoid structure)
#
# H[m,n] = sum_l (a_l/sqrt(nr*nt)) * exp(-j*pi*m*cos(psi_l)) * exp(+j*pi*n*cos(phi_l))
# -- the SAME 2D-sinusoid structure as x=ifft2(Y) in the classical baseline notebook,
# except H is already in that "spatial" domain -- so fft2(H) directly (no ifft2 step)
# reveals peaks at (u=pi*cos(psi), v=pi*cos(phi)).

def _wrap(a): return (a + np.pi) % (2*np.pi) - np.pi

def _bin2angles(k1, k2, N):
    u = _wrap(-2*np.pi*np.asarray(k1)/N)
    v = _wrap( 2*np.pi*np.asarray(k2)/N)
    return np.arccos(np.clip(u/np.pi,-1,1)), np.arccos(np.clip(v/np.pi,-1,1))

def _top_local_maxima(mag, L):
    mx  = maximum_filter(mag, size=3, mode='wrap')
    idx = np.argwhere(mag == mx)
    if len(idx) == 0: return np.zeros((0,2), int)
    vals = mag[idx[:,0], idx[:,1]]
    return idx[np.argsort(-vals)[:L]]

def angles_from_H_sic(H_complex, L, NDFT=1024):
    """2D-DFT + successive interference cancellation, applied directly to H (no ifft2)."""
    r = H_complex.copy()
    m_, n_ = r.shape
    m = np.arange(m_).reshape(-1,1); n = np.arange(n_).reshape(1,-1)
    ks1, ks2 = [], []
    for _ in range(L):
        X = np.fft.fft2(r, s=(NDFT, NDFT))
        k = np.unravel_index(np.argmax(np.abs(X)), X.shape)
        ks1.append(k[0]); ks2.append(k[1])
        u = _wrap(-2*np.pi*k[0]/NDFT); v = _wrap(2*np.pi*k[1]/NDFT)
        a = X[k] / (m_*n_)
        r -= a * np.exp(-1j*m*u) * np.exp(1j*n*v)
    return _bin2angles(ks1, ks2, NDFT)

def evaluate(est_psi, est_phi, true_psi, true_phi, max_deg=1.0):
    L = len(true_psi)
    if len(est_psi) < L: return 0, L, []
    gt = np.stack([true_psi, true_phi], 1)
    es = np.stack([est_psi[:L], est_phi[:L]], 1)
    d = np.linalg.norm(gt[:,None]-es[None], axis=2)
    r, c = linear_sum_assignment(d)
    ndet, good = 0, []
    for i, j in zip(r, c):
        dpsi = np.degrees(np.angle(np.exp(1j*gt[i,0])*np.exp(-1j*es[j,0])))
        dphi = np.degrees(np.angle(np.exp(1j*gt[i,1])*np.exp(-1j*es[j,1])))
        if abs(dpsi) <= max_deg and abs(dphi) <= max_deg:
            ndet += 1; good += [dpsi, dphi]
    return ndet, L, good

print('✅ Angle extraction from H ready')

In [ ]:
# Cell 9 — Stage 2 metric: Pd / RMSE from H_hat (CNN pipeline) vs paper reference
REF_RMSE = {-10:0.553, -5:0.512,  0:0.458,  5:0.392,
             10:0.326,  15:0.279, 20:0.253, 25:0.238}
REF_PD   = {-10:0.204, -5:0.437,  0:0.643,  5:0.779,
             10:0.862,  15:0.899, 20:0.925, 25:0.938}

N_PER_SNR_ANGLE = 500
our_pd, our_rmse = {}, {}

for snr in SNRS_EVAL:
    gen = data_gen_channel(Training=False, condition=(L_EVAL, snr))
    n_det, n_tot, good_all = 0, 0, []
    for _ in range(N_PER_SNR_ANGLE):
        Y_in, H_out, feat = next(gen)
        H_pred = ce_model(tf.expand_dims(Y_in, 0), training=False)[0].numpy()
        H_pred_c = H_pred[...,0] + 1j*H_pred[...,1]

        psi_e, phi_e = angles_from_H_sic(H_pred_c, L_EVAL, NDFT=1024)
        d, t, g = evaluate(psi_e, phi_e, feat[0], feat[1])
        n_det += d; n_tot += t; good_all += g

    our_pd[snr]   = n_det/n_tot
    our_rmse[snr] = np.sqrt(np.mean(np.array(good_all)**2)) if good_all else float('nan')
    print(f'SNR={snr:4d}dB  Pd={our_pd[snr]:.4f}  RMSE={our_rmse[snr]:.4f}  '
          f'(ref Pd={REF_PD[snr]:.3f}, ref RMSE={REF_RMSE[snr]:.3f})')

print('\n✅ Stage-2 angle evaluation complete')

In [ ]:
# Cell 10 — Full comparison table + plots
print('='*88)
print(f'{"SNR":>5} | {"NMSE-CNN(dB)":>13} {"NMSE-LS(dB)":>12} | {"Our Pd":>7} {"Paper Pd":>9} | '
      f'{"Our RMSE":>9} {"Paper RMSE":>10}')
print('-'*88)
for s in SNRS_EVAL:
    print(f'{s:>5} | {10*np.log10(nmse_cnn[s]):>13.2f} {10*np.log10(nmse_ls[s]):>12.2f} | '
          f'{our_pd[s]:>7.4f} {REF_PD[s]:>9.3f} | {our_rmse[s]:>9.4f} {REF_RMSE[s]:>10.3f}')
print('='*88)
print(f'\nParams: this model={my_params:,}  |  paper ResNet={PAPER_RESNET_PARAMS:,}  '
      f'({PAPER_RESNET_PARAMS/my_params:.1f}x smaller)')

fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))

snrs = SNRS_EVAL
axs[0].plot(snrs, [10*np.log10(nmse_cnn[s]) for s in snrs], 'o-', color='teal', label='CNN')
axs[0].plot(snrs, [10*np.log10(nmse_ls[s])  for s in snrs], 's--', color='gray', label='LS (no training)')
axs[0].set_xlabel('SNR (dB)'); axs[0].set_ylabel('NMSE (dB)')
axs[0].set_title('Stage 1: Channel Estimation NMSE'); axs[0].legend(); axs[0].grid(alpha=.3)

axs[1].plot(snrs, [our_pd[s] for s in snrs], 'o-', color='crimson', label='Ours (2-stage)')
axs[1].plot(snrs, [REF_PD[s] for s in snrs], 's--', color='steelblue', label='Paper ResNet (1-stage)')
axs[1].set_xlabel('SNR (dB)'); axs[1].set_ylabel('Pd')
axs[1].set_title('Stage 2: Detection Probability'); axs[1].legend(); axs[1].grid(alpha=.3)

axs[2].plot(snrs, [our_rmse[s] for s in snrs], 'o-', color='crimson', label='Ours (2-stage)')
axs[2].plot(snrs, [REF_RMSE[s] for s in snrs], 's--', color='steelblue', label='Paper ResNet (1-stage)')
axs[2].set_xlabel('SNR (dB)'); axs[2].set_ylabel('RMSE (deg)')
axs[2].set_title('Stage 2: RMSE'); axs[2].legend(); axs[2].grid(alpha=.3)

plt.suptitle(f'Two-stage (channel-est → classical extraction), {my_params:,} params '
             f'vs paper {PAPER_RESNET_PARAMS:,} params', y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'two_stage_comparison.png'), dpi=140, bbox_inches='tight')
plt.show()
print('Saved: two_stage_comparison.png')

In [ ]:
# Cell 11 — Save all results
results = {
    'model': ce_model.name,
    'params': my_params,
    'paper_resnet_params': PAPER_RESNET_PARAMS,
    'param_ratio': my_params / PAPER_RESNET_PARAMS,
    'epochs_trained': len(history.history['loss']),
    'best_loss': float(min(history.history['loss'])),
    'nmse_cnn_db': {str(k): float(10*np.log10(v)) for k,v in nmse_cnn.items()},
    'nmse_ls_db':  {str(k): float(10*np.log10(v)) for k,v in nmse_ls.items()},
    'our_pd':   {str(k): float(v) for k,v in our_pd.items()},
    'our_rmse': {str(k): float(v) for k,v in our_rmse.items()},
    'ref_pd':   {str(k): float(v) for k,v in REF_PD.items()},
    'ref_rmse': {str(k): float(v) for k,v in REF_RMSE.items()},
}
with open(os.path.join(OUT_DIR, 'two_stage_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print('✅ Saved results:')
for fn in ['channel_est_best.weights.h5', 'two_stage_results.json',
           'two_stage_comparison.png', 'channel_est_training_curve.png',
           'channel_est_train_log.csv']:
    p = os.path.join(OUT_DIR, fn)
    sz = os.path.getsize(p)/1e6 if os.path.exists(p) else 0
    print(f'  {fn}  ({sz:.2f} MB)')